In [1]:
import logging
import time
import os
import pickle

#from DataPrep import data_prep

import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [2]:
with open("Pichia_TrTs_2Target.pkl", "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']

In [3]:
Settings = pd.read_csv('../BO_forHyperParameter/Arch1/Round3.csv').iloc[:, 1:]
Settings

,Enc hidden size,Enc Embedding size,Dec Embedding size,Dense Layer size,Dense Layer size aa,Drop rate,Drop rate aa
0,506.0,100.0,59.0,55.0,109.0,0.7,0.1
1,510.0,42.0,224.0,125.0,139.0,0.0,0.7
2,468.0,40.0,43.0,122.0,240.0,0.5,0.8


#### Network parameters

In [4]:
for i in np.arange(0, 3):
    Setting_no = i

    Max_length = 1000
    learning_rate = 0.001
    batch_size = 150
    epochs = 100
    aa_vocab_size = 25
    dna_vocab_size = 67


    hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
    hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
    embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
    embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
    Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
    Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

    drop_rate = Settings['Drop rate'][Setting_no]
    drop_rate_aa = Settings['Drop rate aa'][Setting_no]

    input_sequence = Input(shape=(Max_length,))
    encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
    embedding = encod_emb(input_sequence)

    encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                            merge_mode="concat", weights=None)

    encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

    encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])


    
    decoder_inputs = Input(shape=(Max_length -1, ))
    decoder_inputs_aa = Input(shape=(Max_length, ))

    dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


    final_dex= dex(decoder_inputs)
    final_dex_aa =  encod_emb(decoder_inputs_aa)


    decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
    decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

    decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
    decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


    attn_layer = Attention()
    attn_out = attn_layer([decoder_sequence, encoder_sequence])
    attn_out_aa = attn_layer([decoder_sequence_aa, encoder_sequence])

    decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
    decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


    Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
    Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

    Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
    Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


    dropout_layer = Dropout(drop_rate)
    dropout_output = dropout_layer(Intemediate_output)

    dropout_layer_aa = Dropout(drop_rate_aa)
    dropout_output_aa = dropout_layer(Intemediate_output_aa)

    dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
    logits = dense_layer(dropout_output)

    dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
    logits_aa = dense_layer_aa(dropout_output_aa)

    enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

    enc_dec_model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate = learning_rate),
                  metrics=['accuracy'])
    enc_dec_model.summary()
    
    checkpoint_path = "/Desktop/training_2Target/Round3/" + str(Setting_no) + "/cp.ckpt"
    checkpoint_dir = os.path.dirname(checkpoint_path)

    # Create a callback that saves the model's weights
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
        verbose=0, mode="auto", baseline=None, restore_best_weights=False)
    ## Train the model
    model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                      [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                      batch_size= batch_size, 
                                      epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])
    
    name_history = 'Loss_Evolution/Round3/Combo'+ str(Setting_no) + '.csv'
    pd.DataFrame(model_results.history).to_csv(name_history)
    
    name_model = 'saved_models/Round3/EncDec_' + str(Setting_no)
    enc_dec_model.save(name_model)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1000, 100)    2500        ['input_1[0][0]',                
                                                                  'input_3[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 1000, 1012)  1845888     ['embedding[0][0]']          

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4791 - time_distributed_2_loss: 0.4774 - time_distributed_3_loss: 0.0018 - time_distributed_2_accuracy: 0.4694 - time_distributed_3_accuracy: 0.9998
Epoch 5: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/133 [==============================] - 98s 734ms/step - loss: 0.4791 - time_distributed_2_loss: 0.4774 - time_distributed_3_loss: 0.0018 - time_distributed_2_accuracy: 0.4694 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4613 - val_time_distributed_2_loss: 0.4608 - val_time_distributed_3_loss: 5.5634e-04 - val_time_distributed_2_accuracy: 0.4744 - val_time_distributed_3_accuracy: 0.9998
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4753 - time_distributed_2_loss: 0.4739 - time_distributed_3_loss: 0.0014 - time_distributed_2_accuracy: 0.4703 - time_distributed_3_accuracy: 0.9998
Epoch 6: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/133 [===========

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4651 - time_distributed_2_loss: 0.4644 - time_distributed_3_loss: 6.3182e-04 - time_distributed_2_accuracy: 0.4743 - time_distributed_3_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/133 [==============================] - 96s 719ms/step - loss: 0.4651 - time_distributed_2_loss: 0.4644 - time_distributed_3_loss: 6.3182e-04 - time_distributed_2_accuracy: 0.4743 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4571 - val_time_distributed_2_loss: 0.4568 - val_time_distributed_3_loss: 3.8554e-04 - val_time_distributed_2_accuracy: 0.4820 - val_time_distributed_3_accuracy: 0.9999
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4648 - time_distributed_2_loss: 0.4642 - time_distributed_3_loss: 6.1861e-04 - time_distributed_2_accuracy: 0.4745 - time_distributed_3_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.4599 - time_distributed_2_loss: 0.4595 - time_distributed_3_loss: 4.8591e-04 - time_distributed_2_accuracy: 0.4825 - time_distributed_3_accuracy: 0.9999
Epoch 29: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/133 [==============================] - 96s 720ms/step - loss: 0.4599 - time_distributed_2_loss: 0.4595 - time_distributed_3_loss: 4.8591e-04 - time_distributed_2_accuracy: 0.4825 - time_distributed_3_accuracy: 0.9999 - val_loss: 0.4525 - val_time_distributed_2_loss: 0.4522 - val_time_distributed_3_loss: 3.7330e-04 - val_time_distributed_2_accuracy: 0.4926 - val_time_distributed_3_accuracy: 0.9999
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.4593 - time_distributed_2_loss: 0.4589 - time_distributed_3_loss: 4.6443e-04 - time_distributed_2_accuracy: 0.4838 - time_distributed_3_accuracy: 0.9999
Epoch 30: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/

Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.4355 - time_distributed_2_loss: 0.4349 - time_distributed_3_loss: 5.5185e-04 - time_distributed_2_accuracy: 0.5214 - time_distributed_3_accuracy: 0.9998
Epoch 41: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/133 [==============================] - 95s 714ms/step - loss: 0.4355 - time_distributed_2_loss: 0.4349 - time_distributed_3_loss: 5.5185e-04 - time_distributed_2_accuracy: 0.5214 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4258 - val_time_distributed_2_loss: 0.4254 - val_time_distributed_3_loss: 4.3620e-04 - val_time_distributed_2_accuracy: 0.5396 - val_time_distributed_3_accuracy: 0.9999
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.4307 - time_distributed_2_loss: 0.4302 - time_distributed_3_loss: 4.6031e-04 - time_distributed_2_accuracy: 0.5278 - time_distributed_3_accuracy: 0.9999
Epoch 42: saving model to /Desktop/training_2Target/Round3/0\cp.ckpt
133/

...

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4592 - time_distributed_10_loss: 0.4581 - time_distributed_11_loss: 0.0011 - time_distributed_10_accuracy: 0.4836 - time_distributed_11_accuracy: 0.9996
Epoch 5: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt
133/133 [==============================] - 87s 652ms/step - loss: 0.4592 - time_distributed_10_loss: 0.4581 - time_distributed_11_loss: 0.0011 - time_distributed_10_accuracy: 0.4836 - time_distributed_11_accuracy: 0.9996 - val_loss: 0.4560 - val_time_distributed_10_loss: 0.4552 - val_time_distributed_11_loss: 7.1768e-04 - val_time_distributed_10_accuracy: 0.4870 - val_time_distributed_11_accuracy: 0.9997
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4568 - time_distributed_10_loss: 0.4560 - time_distributed_11_loss: 7.5279e-04 - time_distributed_10_accuracy: 0.4859 - time_distributed_11_accuracy: 0.9997
Epoch 6: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt


Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4469 - time_distributed_10_loss: 0.4466 - time_distributed_11_loss: 3.1837e-04 - time_distributed_10_accuracy: 0.5019 - time_distributed_11_accuracy: 0.9999
Epoch 17: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt
133/133 [==============================] - 85s 642ms/step - loss: 0.4469 - time_distributed_10_loss: 0.4466 - time_distributed_11_loss: 3.1837e-04 - time_distributed_10_accuracy: 0.5019 - time_distributed_11_accuracy: 0.9999 - val_loss: 0.4460 - val_time_distributed_10_loss: 0.4457 - val_time_distributed_11_loss: 2.8205e-04 - val_time_distributed_10_accuracy: 0.5023 - val_time_distributed_11_accuracy: 0.9999
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4460 - time_distributed_10_loss: 0.4457 - time_distributed_11_loss: 2.8676e-04 - time_distributed_10_accuracy: 0.5035 - time_distributed_11_accuracy: 0.9999
Epoch 18: saving model to /Desktop/training_2Target/Round

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.3952 - time_distributed_10_loss: 0.3948 - time_distributed_11_loss: 3.8267e-04 - time_distributed_10_accuracy: 0.5935 - time_distributed_11_accuracy: 0.9998
Epoch 29: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt
133/133 [==============================] - 85s 641ms/step - loss: 0.3952 - time_distributed_10_loss: 0.3948 - time_distributed_11_loss: 3.8267e-04 - time_distributed_10_accuracy: 0.5935 - time_distributed_11_accuracy: 0.9998 - val_loss: 0.4072 - val_time_distributed_10_loss: 0.4067 - val_time_distributed_11_loss: 4.3175e-04 - val_time_distributed_10_accuracy: 0.5756 - val_time_distributed_11_accuracy: 0.9998
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.3803 - time_distributed_10_loss: 0.3799 - time_distributed_11_loss: 3.7709e-04 - time_distributed_10_accuracy: 0.6153 - time_distributed_11_accuracy: 0.9998
Epoch 30: saving model to /Desktop/training_2Target/Round

Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.2173 - time_distributed_10_loss: 0.2170 - time_distributed_11_loss: 3.2216e-04 - time_distributed_10_accuracy: 0.8132 - time_distributed_11_accuracy: 0.9999
Epoch 41: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt
133/133 [==============================] - 86s 645ms/step - loss: 0.2173 - time_distributed_10_loss: 0.2170 - time_distributed_11_loss: 3.2216e-04 - time_distributed_10_accuracy: 0.8132 - time_distributed_11_accuracy: 0.9999 - val_loss: 0.2968 - val_time_distributed_10_loss: 0.2964 - val_time_distributed_11_loss: 3.6188e-04 - val_time_distributed_10_accuracy: 0.7541 - val_time_distributed_11_accuracy: 0.9999
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.2103 - time_distributed_10_loss: 0.2096 - time_distributed_11_loss: 7.6829e-04 - time_distributed_10_accuracy: 0.8209 - time_distributed_11_accuracy: 0.9997
Epoch 42: saving model to /Desktop/training_2Target/Round

Epoch 53/100
133/133 [==============================] - ETA: 0s - loss: 0.1481 - time_distributed_10_loss: 0.1478 - time_distributed_11_loss: 2.8137e-04 - time_distributed_10_accuracy: 0.8819 - time_distributed_11_accuracy: 0.9999
Epoch 53: saving model to /Desktop/training_2Target/Round3/2\cp.ckpt
133/133 [==============================] - 86s 644ms/step - loss: 0.1481 - time_distributed_10_loss: 0.1478 - time_distributed_11_loss: 2.8137e-04 - time_distributed_10_accuracy: 0.8819 - time_distributed_11_accuracy: 0.9999 - val_loss: 0.2669 - val_time_distributed_10_loss: 0.2666 - val_time_distributed_11_loss: 3.6222e-04 - val_time_distributed_10_accuracy: 0.8141 - val_time_distributed_11_accuracy: 0.9999
Epoch 54/100
133/133 [==============================] - ETA: 0s - loss: 0.1447 - time_distributed_10_loss: 0.1444 - time_distributed_11_loss: 2.9656e-04 - time_distributed_10_accuracy: 0.8850 - time_distributed_11_accuracy: 0.9999
Epoch 54: saving model to /Desktop/training_2Target/Round

In [ ]:
name_model = 'saved_models/Round3/EncDec_' + str(Setting_no)
enc_dec_model.save(name_model)

#### Length and its plot

In [ ]:
length_aa = []
for i in range(len(AA_seq_tokenized)):
    length_aa.append(len(AA_seq_tokenized[i]))
print(max(length_aa))

sns.histplot(length_aa)


In [ ]:
 Cds_ts[0,0]

In [ ]:
Setting_no